# Week 2 practical: gateway contracts with a fake backend

This notebook is the first runnable Week 2 lab. It teaches the gateway before we connect a real model server: validate a public request, protect capacity, map a backend result, map a timeout, and relay stream chunks.

**Sources and attribution:** see `../NOTEBOOK-SOURCES.md`. The flow is original for this repository, informed by Abi Aryan's Class 2 learning arc (server → gateway → failure → observability) and the Week 2 reading plan in `ROADMAP.md`.

**Success condition:** every assertion passes; then you can replace the fake with an HTTP adapter to the Week 1 server.

In [1]:
import asyncio
import json
import uuid
from dataclasses import dataclass, field
from typing import AsyncIterator, Literal

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

MAX_OUTPUT_TOKENS = 128
print('Gateway lab ready; max output tokens =', MAX_OUTPUT_TOKENS)

Gateway lab ready; max output tokens = 128


/Users/myatkaung/Desktop/production-llm-inference-lab/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Define the public contract and private backend contract

The client knows `max_tokens`. The Week 1 backend knows `max_new_tokens`. Keeping these models separate makes the translation visible and testable.

In [2]:
class ChatMessage(BaseModel):
    role: Literal['system', 'user', 'assistant']
    content: str = Field(min_length=1)

class ChatCompletionRequest(BaseModel):
    model: str = Field(min_length=1)
    messages: list[ChatMessage] = Field(min_length=1)
    max_tokens: int = Field(default=64, ge=1)
    stream: bool = False

class BackendGenerateRequest(BaseModel):
    messages: list[ChatMessage]
    max_new_tokens: int

@dataclass
class BackendResult:
    text: str
    prompt_tokens: int
    output_tokens: int
    backend_model_id: str = 'fake-qwen'

class BackendTimeout(Exception):
    pass

def to_backend_request(request: ChatCompletionRequest) -> BackendGenerateRequest:
    return BackendGenerateRequest(messages=request.messages, max_new_tokens=request.max_tokens)

sample = ChatCompletionRequest(model='local-qwen', messages=[{'role': 'user', 'content': 'What is caching?'}], max_tokens=16)
print(to_backend_request(sample).model_dump())

{'messages': [{'role': 'user', 'content': 'What is caching?'}], 'max_new_tokens': 16}


## 2. Build a fake backend

This object does not run a model. It records calls and returns controlled outcomes. That lets a test prove whether the gateway itself behaves correctly.

In [3]:
@dataclass
class FakeBackend:
    result: BackendResult = field(default_factory=lambda: BackendResult('Caching stores reusable data.', 11, 5))
    error: Exception | None = None
    chunks: list[str] = field(default_factory=lambda: ['C', 'aching'])
    calls: list[BackendGenerateRequest] = field(default_factory=list)

    async def generate(self, request: BackendGenerateRequest) -> BackendResult:
        self.calls.append(request)
        if self.error is not None:
            raise self.error
        return self.result

    async def stream(self, request: BackendGenerateRequest) -> AsyncIterator[str]:
        self.calls.append(request)
        if self.error is not None:
            raise self.error
        for chunk in self.chunks:
            yield chunk

## 3. Build the smallest gateway around that interface

The route has five steps: policy check, request ID, translation, backend call, public response mapping.

In [4]:
def public_response(result: BackendResult, request_id: str, model: str):
    return {
        'id': request_id, 'object': 'chat.completion', 'model': model,
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': result.text}, 'finish_reason': 'stop'}],
        'usage': {'prompt_tokens': result.prompt_tokens, 'completion_tokens': result.output_tokens, 'total_tokens': result.prompt_tokens + result.output_tokens},
    }

def build_app(backend: FakeBackend) -> FastAPI:
    app = FastAPI()

    @app.post('/v1/chat/completions')
    async def chat_completions(request: ChatCompletionRequest):
        request_id = f'req_{uuid.uuid4().hex[:12]}'
        if request.max_tokens > MAX_OUTPUT_TOKENS:
            raise HTTPException(400, detail={'type': 'output_token_limit', 'request_id': request_id})
        if not request.stream:
            try:
                result = await backend.generate(to_backend_request(request))
            except BackendTimeout as exc:
                raise HTTPException(504, detail={'type': 'backend_timeout', 'request_id': request_id, 'message': str(exc)}) from exc
            return public_response(result, request_id, request.model)
        async def events():
            async for text in backend.stream(to_backend_request(request)):
                payload = {'id': request_id, 'choices': [{'index': 0, 'delta': {'content': text}}]}
                yield 'data: ' + json.dumps(payload) + chr(10) + chr(10)
            yield 'data: [DONE]' + chr(10) + chr(10)
        return StreamingResponse(events(), media_type='text/event-stream')

    return app

## 4. Practical test A: success mapping

This proves that a known backend result becomes a stable public response. It does not depend on Qwen or MPS.

In [5]:
backend = FakeBackend()
client = TestClient(build_app(backend))
response = client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'What is caching?'}], 'max_tokens': 16})
assert response.status_code == 200
body = response.json()
assert body['choices'][0]['message']['content'] == 'Caching stores reusable data.'
assert body['usage'] == {'prompt_tokens': 11, 'completion_tokens': 5, 'total_tokens': 16}
assert body['id'].startswith('req_')
assert len(backend.calls) == 1
print(body)

{'id': 'req_ad53f694729a', 'object': 'chat.completion', 'model': 'local-qwen', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Caching stores reusable data.'}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 11, 'completion_tokens': 5, 'total_tokens': 16}}


## 5. Practical test B: protect backend capacity

This is the key fake-backend assertion: reject a valid-looking but over-limit request before it consumes inference capacity.

In [6]:
backend = FakeBackend()
client = TestClient(build_app(backend))
response = client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'Hello'}], 'max_tokens': 500})
assert response.status_code == 400
assert response.json()['detail']['type'] == 'output_token_limit'
assert response.json()['detail']['request_id'].startswith('req_')
assert backend.calls == []
print('rejected before backend call:', response.json())

rejected before backend call: {'detail': {'type': 'output_token_limit', 'request_id': 'req_5369e7dad5af'}}


## 6. Practical test C: controlled timeout and streaming

A fake lets us force an error immediately instead of sleeping for a real 30-second deadline. Then it lets us verify ordered stream chunks and `[DONE]`.

In [7]:
timeout_backend = FakeBackend(error=BackendTimeout('simulated timeout'))
timeout_client = TestClient(build_app(timeout_backend))
timeout_response = timeout_client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'Hello'}], 'max_tokens': 16})
assert timeout_response.status_code == 504
assert timeout_response.json()['detail']['type'] == 'backend_timeout'

stream_backend = FakeBackend(chunks=['C', 'aching'])
stream_client = TestClient(build_app(stream_backend))
stream_response = stream_client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'Hello'}], 'max_tokens': 16, 'stream': True})
assert stream_response.status_code == 200
events = [line.removeprefix('data: ') for line in stream_response.text.splitlines() if line.startswith('data: ')]
payloads = [json.loads(event) for event in events[:-1]]
assert [payload['choices'][0]['delta']['content'] for payload in payloads] == ['C', 'aching']
assert events[-1] == '[DONE]'
print('timeout:', timeout_response.json())
print('stream:', stream_response.text)

timeout: {'detail': {'type': 'backend_timeout', 'request_id': 'req_fd57ef5b5555', 'message': 'simulated timeout'}}
stream: data: {"id": "req_2aa5e4b53ecc", "choices": [{"index": 0, "delta": {"content": "C"}}]}

data: {"id": "req_2aa5e4b53ecc", "choices": [{"index": 0, "delta": {"content": "aching"}}]}

data: [DONE]




## 7. Your next practical change

1. Change the fake result's `prompt_tokens` and `output_tokens`; rerun Test A and predict the new `total_tokens`.
2. Add a fake backend error of your own (for example a generic backend failure) and design the gateway status/code you want.
3. Only after these tests are understood, implement `HttpWeek1Backend` in the Week 2 source folder and run one real request through port 8000.

**Reflection:** Which test here proves gateway behavior, and which future test will prove real HTTP integration?